In [4]:
import cv2
import numpy as np

very helpful article https://sachithppp.medium.com/projection-histogram-of-binary-image-e4e9eabb7c9d

In [14]:
im = cv2.imread('captures/Alice.JPG', cv2.IMREAD_GRAYSCALE)
GaussianFilter= cv2.GaussianBlur(im, (5,5), 0)

In [15]:
thret, binarizedImage = cv2.threshold(GaussianFilter, 220, 255, cv2.THRESH_BINARY)
#all colours above 220 are put to 255 and all below to 0

In [16]:
cv2.imwrite('binImg.png', binarizedImage)

True

In [17]:
binarizedImage[binarizedImage == 0] = 1
binarizedImage[binarizedImage == 255] = 0

In [18]:
horizontal_projection = np.sum(binarizedImage, axis=1);
print(horizontal_projection)

[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
  44  58  62  65  57  59  56  56  55  53  59  58  56  36   0   0   0   0
   0   0   0  12  43  53  60 112 130 135 136 127 121 122 126 120  87  10
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   7  43 164 223 253 257 255 247 191  72
  34   7  27 130 227 270 275 269 263 228  72  37   3  22  94 206 264 267
 272 273 252  92  29   4  25  64 142 226 240 242 242 234 174  46   4   0
   0   0   0   0   0   0   0   0   0   0   0  25  68 228 263 273 270 265
 260 182  43   8  11  54 126 250 258 253 252 249 233 146  45  16  31  73
 227 278 288 288 286 282 251 103  46  14  76 126 267 280 281 281 279 275
 216  73  14   0   0   0   0   0   0   0   0   0   0   0   0  30  92 183
 265 272 270 269 267 261 209  53   6  57 115 226 270 273 268 269 262 257
 184  29   4  38  91 231 277 278 281 279 276 268 169   8   4  38  93 261
 280 284 282 278 279 269 161   7  34 127 177 274 28

In [19]:
height, width = binarizedImage.shape
print('width : ', width)
print('height : ', height)
blankImage = np.zeros((height, width, 3), np.uint8)
for row in range(height):
    cv2.line(blankImage, (0,row), (int(horizontal_projection[row]*width/height),row), (255,255,255), 1)
cv2.imwrite('plot.png', blankImage)

width :  318
height :  434


True

In [20]:
def find_break(hoz_prj,empty_rows):
    zeros = 0
    splitx = []
    row = 0
    detected=False
    for i in hoz_prj:
        if row == 0 and i == 0:
            detected = True
            zeros = zeros + 1
        elif i == 0:
            zeros = zeros + 1
        if zeros == empty_rows and detected == False:
            splitx.append(row)
            detected=True
            zeros = 0
        elif detected == True and i != 0:
            detected = False
            zeros = 0
        row = row+1
    return splitx   

In [21]:
a = find_break(horizontal_projection,15)

for i in a:
    print(i)

61
185
353
430


Using position of empty spaces we can crop the original unfiltered image

In [22]:
height, width = im.shape
cropped_img = im[0:58, 0:width]
cv2.imwrite('crop1.png', cropped_img)

True

In [23]:
len(a)

4

In [24]:
def get_crops(im, breaks,height,width):
    for i in range(len(breaks)):
        if i == 0:
            cropped_img = im[0:breaks[i], 0:width]
            cv2.imwrite('output/HorizontalCrops/crop_im_'+str(i)+'.png', cropped_img)
        elif i == len(breaks):
            cropped_img = im[breaks[i]:height, 0:width]
            cv2.imwrite('output/HorizontalCrops/crop_im_'+str(i)+'.png', cropped_img)
        else:
            cropped_img = im[breaks[i-1]:breaks[i], 0:width]
            cv2.imwrite('output/HorizontalCrops/crop_im_'+str(i)+'.png', cropped_img)

In [25]:
get_crops(im,a,318,434)

In [26]:
im

array([[226, 226, 226, ..., 244, 244, 244],
       [226, 226, 226, ..., 244, 244, 244],
       [226, 226, 226, ..., 244, 244, 244],
       ...,
       [222, 222, 222, ..., 230, 229, 229],
       [221, 221, 221, ..., 229, 229, 229],
       [221, 221, 221, ..., 229, 229, 229]], shape=(434, 318), dtype=uint8)